In [1]:
import sys
from datetime import datetime
import numpy as np
def print_env():
    print('====================================', datetime.now().strftime('%Y-%m-%d %H:%M:%S'), '====================================')
    !cat /proc/cpuinfo  | grep 'model name'| uniq
    !cat /proc/cpuinfo  | grep 'cpu family' | uniq
    !cat /proc/cpuinfo  | grep 'model' | grep -v 'name' | uniq
    !cat /proc/cpuinfo  | grep 'MHz'| uniq
    !lscpu | grep '^CPU(s):'| uniq
    print('python:', sys.version)
    print('numpy:', np.__version__)
    print('====================================', '===================', '====================================')
print_env()  

==================================== 2026-10-10 07:16:55 ====================================
model name	: Intel Xeon Processor (Cascadelake)
cpu family	: 6
model		: 85
cpu MHz		: 2099.998
CPU(s):                                  8
python: 3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]
numpy: 2.3.5
==================================== =================== ====================================


In [2]:
import numpy as np
import tensorflow as tf
import time
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category = ConvergenceWarning) # suppress convergence warnings

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data() # load MNIST train and test
n_features = X_train.shape[1] * X_train.shape[2]    # number of features: 784
n_train, n_test = X_train.shape[0], X_test.shape[0] # train: 60,000 / test: 10,000
X_train = X_train.reshape(n_train, n_features).astype('float32')  # one sample per row
X_test = X_test.reshape(n_test, n_features).astype('float32')     # one sample per row

2026-10-10 07:16:57.138551: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
# Rosenblatt's Perceptron Algorithm
def perceptron_algorithm(X, y, max_iter=100, verbose=False):
    y_ = np.where(y == 1, 1, -1)
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    b = 0.0
    for iter in range(max_iter):
        changed = False
        for i in range(n_samples):
            if y_[i] * (np.dot(w, X[i]) + b) <= 0:
                w += y_[i] * X[i]
                b += y_[i]
                changed = True
        if verbose:
            print(f'  epoch: {iter}/{max_iter}')
        if not changed:
            break
    return w, b

In [4]:
# return (normalized) normal vector and bias of the perpendicular bisector of two points
def perpendicular_bisector(pos, neg):
    norm = pos - neg
    mid = (pos + neg) / 2
    bias = -np.dot(norm, mid)
    return norm, bias

In [5]:
# Oblique Decision Tree node (decision or leaf)
class DT_node:
    def __init__(self, w=None, b=None, label=None, samples=0):
        self.w, self.b = w, b
        self.label = label # None if mixed
        self.samples = samples
    def add_left_child(self, child):
        self.left = child
    def add_right_child(self, child):
        self.right = child

In [6]:
# Oblique Decision Tree
class DT:
    def __init__(self, root_node):
        
        self.root = root_node
        self.all_nodes, self.decision_nodes, self.leaf_nodes = 0, 0, 0 

        # DFS => 
        # - count all_nodes, decision_nodes, leaf_nodes
        # - assign node.first_layer_index (visit time) to decision nodes
        # - collect all coeffs, intercepts from decision nodes, form matrices
        # - collect path patterns to leaf nodes per class (class -> [paths...])
        def visit(node, first_layer_index, path):
            
            #  =========== leaf node ===========
            if (node.label is not None): 
                
                if node.label in patterns:
                   patterns[node.label].append(path.copy()) # append to existing list
                else:
                    patterns[node.label] = [path.copy()] # list of one list
                return first_layer_index # return unchanged first_layer_index
                
            #  =========== decision node ===========                
            else:
                
                node.first_layer_index = first_layer_index # index: next available index
                
                coeffs.append(node.w) 
                intercepts.append(node.b)

                path.append((node.first_layer_index, -1)) # left: -1
                first_layer_index = visit(node.left, first_layer_index + 1, path) # returns index allocated by this node (if did)
                path.pop()
                
                path.append((node.first_layer_index, +1)) # right: +1
                first_layer_index = visit(node.right, first_layer_index, path) # index returned from left visit
                path.pop()
                
                return first_layer_index

        coeffs, intercepts = [], [] # of decision nodes, used temporarily
        patterns = {} # path patterns to leaf nodes per class, used temporarily
        path = []
        
        self.decision_nodes = visit(root_node, 0, path)
        self.leaf_nodes = self.decision_nodes + 1
        self.all_nodes = self.decision_nodes + self.leaf_nodes

        self.first_layer_size = self.decision_nodes
        self.second_layer_size = self.leaf_nodes
        self.third_layer_size = 0 # number of classes (actually used)
        
        self.W1, self.b1 = np.vstack(coeffs), np.array(intercepts)

        self.third_layer_index_to_class_mapping = {} # output neuron index -> class label
        self.construct_second_and_third_layer(patterns)
        
        del coeffs, intercepts, path, patterns

    # create a list of neurons for the second layer, also create the corresponding output neurons        
    def construct_second_and_third_layer(self, patterns): 
        
        self.second_layer_size = self.leaf_nodes
        
        second_layer_weights, second_layer_biases = [], []
        third_layer_weights, third_layer_biases = [], [] 
        second_layer_start_index = 0

        for label in sorted(patterns): # classes, ordered

            # create second layer neurons (one for each leaf of the respective class, batch)
            class_weights, class_biases = [], []
            
            # patterns class -> [[tuple1, tuple2], [tuple1, tuple3], ...] where a tuple: (node index,outcome -1/1) 
            for pattern in patterns[label]:
                
                w = self.pattern_to_weights(pattern) 
                b = -(w == 1).sum() + 0.5
                class_weights.append(w)
                class_biases.append(b)
                
            second_layer_weights.append(class_weights)    
            second_layer_biases.extend(class_biases)
            n_class_neurons = len(class_weights)
            
            # create output neruon for this class (connections back to a batch of second-layer neurons)
            output_neuron_weights = np.zeros(self.second_layer_size, dtype=np.int8)
            output_neuron_weights[second_layer_start_index:second_layer_start_index + n_class_neurons] = 1
            second_layer_start_index += n_class_neurons
            
            third_layer_weights.append(output_neuron_weights)
            third_layer_biases.append(-0.5)

            # register output neuron index -> class label mapping for later predicion
            third_layer_index = len(third_layer_weights) - 1
            self.third_layer_index_to_class_mapping[third_layer_index] = label

        self.W2, self.b2 = np.vstack(second_layer_weights), np.array(second_layer_biases)
        self.W3, self.b3 = np.vstack(third_layer_weights), np.array(third_layer_biases)
        self.third_layer_size = self.W3.shape[0]

    # list to sparse representation
    def pattern_to_weights(self, pattern): 
        weights = np.zeros(self.first_layer_size, dtype=np.int8)
        for (first_layer_index, outcome) in pattern:
            weights[first_layer_index] = outcome
        return weights

    def first_layer(self):
        return self.W1, self.b1

    def second_layer(self):
        return self.W2, self.b2

    def third_layer(self):
        return self.W3, self.b3

    def print_stats(self):
        print(f'  nodes: {self.all_nodes} (decisions: {self.decision_nodes}, leafs: {self.leaf_nodes}), classes: {self.third_layer_size})')

In [7]:
def split_multiclass(X, y, method, bias_optimization):
    
    classes = np.unique(y)
    if classes.size == 1: 
        return DT_node(label=classes[0], samples=X.shape[0])
        
    best_score = -1
    best_label = -1

    for c in classes:
        y_c = np.where(y == c, True, False) # relabel 1-vs-rest

        if method == 'lr':
            model = LogisticRegression() # seperate c the best from the rest
            model.fit(X, y_c)
            w_c = model.coef_[0]
            b_c = model.intercept_[0]
        elif method == 'lda':
            if X.shape[0] == 2: # to avoid LDA ValueError: The number of samples must be more than the number of classes.
                w_c, b_c = perpendicular_bisector(X[0], X[1])
            else:
                model = LinearDiscriminantAnalysis()
                model.fit(X, y_c)
                w_c = model.coef_[0]
                b_c = model.intercept_[0]            
        elif method == 'svc':
            model = SVC(kernel='linear', max_iter=1000)            
            model.fit(X, y_c)
            w_c = model.coef_[0]
            b_c = model.intercept_[0]
        elif method == 'perceptron':
            w_c, b_c = perceptron_algorithm(X, y)
        else:            
            raise Exception('unknown method')

        # optimize bias to max separation
        if bias_optimization == 'binary':
            b_c_opt, score_c = optimize_bias_multiclass(X, y_c, w_c) # y_c is used
            if score_c > best_score:
                best_score = score_c
                best_label = c
                w, b = w_c, b_c_opt
        elif bias_optimization == 'multi':
            b_c_opt, score_c_opt = optimize_bias_multiclass(X, y, w_c) # y is used
            if score_c_opt > best_score:
                best_score = score_c_opt
                best_label = c
                w, b = w_c, b_c_opt
        else: # no bias optimization
            score_c = score_multiclass(X, y_c, w_c, b_c) # or y?
            if score_c > best_score:
                best_score = score_c
                best_label = c
                w, b = w_c, b_c
            pass 
    
    if best_score == 0: # workaround for Perceptron with no split: optimize bias 
        if bias_optimization == 'none':
            y_c = np.where(y == best_label, True, False) # relabel 1-vs-rest
            b, new_score = optimize_bias_multiclass(X, y_c, w)
        else:
            raise Exception("no split")
    
    decision_node = DT_node(w=w, b=b, samples=X.shape[0])

    # split    
    s = X @ w + b
    pos_side_mask = s >= 0
    X_left, y_left = X[~pos_side_mask], y[~pos_side_mask]
    X_right, y_right = X[pos_side_mask], y[pos_side_mask]
    
    if X_right.shape[0] > 0 and X_left.shape[0] > 0:        
        right_child = split_multiclass(X_right, y_right, method, bias_optimization)
        decision_node.add_right_child(right_child)

        left_child = split_multiclass(X_left, y_left, method, bias_optimization)
        decision_node.add_left_child(left_child)
    
    if X_right.shape[0] == 0 or X_right.shape[0] == 0:
        raise Exception('no split')
    
    return decision_node

In [8]:
def optimize_bias_multiclass(X, y, w): # returns b and score
    scores = X @ w
    n = len(y)
    order = np.argsort(scores) # sort by projection onto w
    scores = scores[order]
    y_sorted = y[order]
    n_classes = int(np.max(y)) + 1
    total = np.bincount(y_sorted, minlength=n_classes) # initially everything is on the right
    left = np.zeros(n_classes, dtype=np.int64)
    right = total.copy()
    best_score = 0
    best_b = -scores[0] - 1.0
    i = 0
    while i < n: # process all points having the same projection
        j = i + 1
        while j < n and scores[j] == scores[i]:
            j += 1
        group = np.bincount(y_sorted[i:j], minlength=n_classes) # move this entire group from right to left.
        left += group
        right -= group
        if j < n and scores[j] > scores[i]: # a threshold is possible only if there is a gap after this group.
            score = np.sum(left * (right.sum() - right)) # number of cross-label pairs separated by the threshold
            if score > best_score:
                best_score = int(score)
                threshold = (scores[i] + scores[j]) / 2 # any threshold in (scores[i], scores[j]) is equivalent
                best_b = -threshold
        i = j
    return best_b, best_score

def score_multiclass(X, y, w, b, verbose=False):
    s = X @ w + b
    positive = s >= 0 # positive=right, negative=left
    negative = ~positive
    left_counts = np.bincount(y[negative], minlength=10) # number of samples of each class
    right_counts = np.bincount(y[positive], minlength=10)    
    return int(left_counts.sum() * right_counts.sum() - np.dot(left_counts, right_counts)) # every left sample is paired with every right sample, except pairs having the same label

In [9]:
def predict(X, W, b):
    return ((X @ W.T) + b >= 0).astype(int)

def predict_all(X, W1, b1, W2, b2, W3, b3):
    X1 = predict(X, W1, b1)
    X2 = predict(X1, W2, b2)
    X3 = predict(X2, W3, b3).astype(bool)
    return X3
    
def evaluate(X_train, y_train, X_test, y_test, W1, b1, W2, b2, W3, b3):
    # print(f'---------- Boolean Training set ----------')
    all_predictions = predict_all(X_train, W1, b1, W2, b2, W3, b3)  # bool matrix
    counts = all_predictions.sum(axis=1)
    x = np.argmax(all_predictions, axis=1)
    x[counts != 1] = -1
    print(f'  Training accuracy: {100*np.sum(y_train == x)/X_train.shape[0]:.2f}% ({np.sum(y_train == x)}/{X_train.shape[0]})')
    # print(f'---------- Boolean Test set ----------')
    all_predictions = predict_all(X_test, W1, b1, W2, b2, W3, b3)  # bool matrix
    counts = all_predictions.sum(axis=1)
    x = np.argmax(all_predictions, axis=1)
    x[counts != 1] = -1
    print(f'  Test accuracy: {100*np.sum(y_test == x)/X_test.shape[0]:.2f}% ({np.sum(y_test == x)}/{X_test.shape[0]})')    

In [10]:
np.random.seed(42) # for reproducibility only

X = X_train
y = y_train

separation_methods = ['lda', 'svc', 'lr']
bias_optimizations = ['binary'] # ['none', 'binary', 'multi']

for separation_method in separation_methods:
    for bias_optimization in bias_optimizations:
        start = time.perf_counter()

        # === first layer neurons ===
        root = split_multiclass(X, y, separation_method, bias_optimization)
        # === second and third layers ===
        dt = DT(root)

        elapsed_s = (time.perf_counter() - start)
        print(f'Method: {separation_method} (bias optimization: {bias_optimization})')
        l1, l2, l3 = dt.first_layer_size, dt.second_layer_size, dt.third_layer_size
        features = X.shape[1]
        print(f'  Layer sizes: {l1}-{l2}-{l3}, Total number of neurons: {l1+l2+l3}')
        print(f'  Construction time: {elapsed_s:.2f}s')
        print(f'  Total number of parameters: { (features + 1) * l1 + (l1 + 1) * l2 + (l2 + 1) * l3 }')
        W1, b1 = dt.first_layer()
        W2, b2 = dt.second_layer()
        W3, b3 = dt.third_layer()
        evaluate(X_train, y_train, X_test, y_test, W1, b1, W2, b2, W3, b3) # verify training accuracy, print test accuracy

Method: lda (bias optimization: binary)
  Layer sizes: 625-626-10, Total number of neurons: 1261
  Construction time: 384.19s
  Total number of parameters: 888771
  Training accuracy: 100.00% (60000/60000)
  Test accuracy: 85.63% (8563/10000)
Method: svc (bias optimization: binary)
  Layer sizes: 356-357-10, Total number of neurons: 723
  Construction time: 2070.12s
  Total number of parameters: 410489
  Training accuracy: 100.00% (60000/60000)
  Test accuracy: 89.65% (8965/10000)
Method: lr (bias optimization: binary)
  Layer sizes: 278-279-10, Total number of neurons: 567
  Construction time: 581.75s
  Total number of parameters: 298871
  Training accuracy: 100.00% (60000/60000)
  Test accuracy: 89.77% (8977/10000)


In [11]:
# Method: lda (bias optimization: binary)
#   Layer sizes: 625-626-10, Total number of neurons: 1261
#   Construction time: 384.19s
#   Total number of parameters: 888771
#   Training accuracy: 100.00% (60000/60000)
#   Test accuracy: 85.63% (8563/10000)
# Method: svc (bias optimization: binary)
#   Layer sizes: 356-357-10, Total number of neurons: 723
#   Construction time: 2070.12s
#   Total number of parameters: 410489
#   Training accuracy: 100.00% (60000/60000)
#   Test accuracy: 89.65% (8965/10000)
# Method: lr (bias optimization: binary)
#   Layer sizes: 278-279-10, Total number of neurons: 567
#   Construction time: 581.75s
#   Total number of parameters: 298871
#   Training accuracy: 100.00% (60000/60000)
#   Test accuracy: 89.77% (8977/10000)
# Method: perceptron (bias optimization: binary)
#   Layer sizes: 20977-20978-10, Total number of neurons: 41965
#   Construction time: 892.16s
#   Total number of parameters: 456753219
#   Training accuracy: out of mem?
#   Test accuracy: out of mem?